# **PicassoPy Workshop --> Test case: cpv 2024-08-21**
---


- Folder for data `test_case_data` -> download separately
- Folder for configs `test_case_config`


## Imports

In [ ]:
import os, sys
import argparse
import datetime
import logging
from pathlib import Path
import numpy as np

sys.path.append('../')
import ppcpy
import ppcpy.io.loadConfigs as loadConfigs
import ppcpy.io.readPollyRawData as readPollyRawData
import ppcpy.interface.picassoProc as picassoProc
import ppcpy.misc.helper as helper
import ppcpy.misc.startscreen as startscreen
from ppcpy.io.write2nc import write_channelwise_2_nc_file, write2nc_file, write_profile2nc_file

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.tab20.colors)

## Defining Script Inputs

- The parameters `args.device`, `args.timestamp`, `args.picasso_config_file`, `args.level0_file_to_process`, need to be manually specified per case.
- The parameter `DATABASE_PATH` is the path to the database used for storing the retrieved calibration constants

In [ ]:
## For purpose of the notebook mimic the argparse interface
from types import SimpleNamespace
args = SimpleNamespace()

## The used device and time of mesurment
args.device = 'pollyxt_cpv'
args.timestamp = '20240821'
dt = datetime.datetime.strptime(args.timestamp, "%Y%m%d")

## The used config file
args.picasso_config_file = "test_case_config/pollynet_processing_chain_config_test.json"

## The data file to use
args.level0_file_to_process = f"test_case_data/{dt:%Y_%m_%d_%a}_CPV_00_00_01.nc"


## Database path
DATABASE_PATH = "PicassoPyDatabase.db" # I should test if it works to read this directly from the config files.

In [ ]:
startscreen.startscreen()

## Load Data and Config-files

Loads data and information from the config files into the following three dictionaries:
- `picasso_config_dict`: Paths and other information stored in the picasso config file
- `polly_config_dict`: Configuration variables from polly config and polly default files
- `rawdata_dict`: Measurement data and information extracted from the level0 file

In [ ]:
## Path to dafault Picasso config file
picasso_default_config_file = Path(
    helper.detect_path_type(Path.cwd().parent), 'ppcpy', 'config', 'pollynet_processing_chain_config.json')

## Load Picasso config file
picasso_config_dict = loadConfigs.loadPicassoConfig(args.picasso_config_file, picasso_default_config_file)

## load polly config file
polly_config_array = loadConfigs.readPollyNetConfigLinkTable(picasso_config_dict['pollynet_config_link_file'], timestamp=args.timestamp, device=args.device)
polly_config_dict = loadConfigs.getPollyConfigfromArray(
    polly_config_array, picasso_config_dict
)

## Load level0-data file
rawfile_fullname = args.level0_file_to_process
rawfile = helper.detect_path_type(rawfile_fullname)
rawdata_dict = readPollyRawData.readPollyRawData(rawfile)

## Initialize PicassoProc object

PicassoProc is the main object in the PicassoPy, and is responsible for running all processes included and storing the data.

In [ ]:
## Initialize PicassoProc
data_cube = picassoProc.PicassoProc(rawdata_dict, polly_config_dict, picasso_config_dict)

In [ ]:
## reset date if date in filename differs date within nc-file 
data_cube.reset_date_infile()

## checking for correct mshots
data_cube.check_for_correct_mshots()

## setting channelTags
data_cube.setChannelTags()

## check for correct date in nc-file
data_cube.reset_date_infile()

## Preprocessing & Saturation Detection

The preprocessing includes the following processes:
- Deadtime correction
- Background correction
- SNR claculations
- Flagging of data
- Range correction

In [ ]:
## Perform preprocessing, this includes Dead-time correction, Background correction, and Range correction
data_cube.preprocessing(collect_debug=True)

In [ ]:
## Save high resolution signal-to-noise ratio, background, and range corrected signal
write_channelwise_2_nc_file(data_cube=data_cube, prod_ls=['SNR', 'BG', 'RCS'])

In [ ]:
## Display available channels
data_cube.channel_dict

In [ ]:
## Detect and flag saturated signal
data_cube.SaturationDetect()

## Depol Calibration


- Depol. calibration constants (DC) are retrieved at each depol. calibration period included in the data
- All retrieved DCs are stored in a dedicated database
- The optimal retrieved DC, ie. the one with the lowest standard deviation (std) is used for the processing
- If no DCs can be retirieved, the DC with the lowest std in the time range [24h before the measurement, 24h after the measurement] included in the database will be used

In [ ]:
## Delta 90 polarization calibration
data_cube.polarizationCaliD90()

In [ ]:
## Display depolarization calibration constants
data_cube.etaused

## Cloud Screening

Three modes of cloud Screening are currently implemented:

0. No cloud screening. Return cloud free for all timestamps
1. Cloud screen with Maximum Gradiant Signal (MSG) algorithm
2. Cloud screen with Zhao's algorithm

Clouds are screened per timestamp (30s). After the screening the data is splitt up into cloud free segments and aggregated.

In [ ]:
## Apply cloud screening
data_cube.cloudScreen()

In [ ]:
## Segmentate cloud free groups
data_cube.cloudFreeSeg()

In [ ]:
## Display cloud free groups
data_cube.clFreeGrps

In [ ]:
## Aggregate background, background corrected signal, and range corrected signal
data_cube.aggregate_profiles()

## Molecular Profiles

The molecular profiles are calculated from cloudNet ECMWF model data. Gdas1 data is not supported in PicassoPy!

In [ ]:
## QuickFix for loadMeteo bug:
METEO_DATA_DIR_PATH = "E:\\data\\level1a\\cloudnet\\mindelo\\calibrated\\ecmwf"
data_cube.polly_config_dict['meteorDataSource'] = 'nc_cloudnet'
data_cube.polly_config_dict['meteo_folder'] = METEO_DATA_DIR_PATH
data_cube.polly_config_dict['meteo_file'] = r"[\\/]{0:%Y}[\\/]{0:%Y%m%d}_.*\.nc"

## Load meteorological data
data_cube.loadMeteo()

## Calculate molecular profiles
data_cube.calcMolecular()

## Rayleigh-Fit

- Douglas-Peucker algorithm is used to segment the signal into potential reference heights
- The reference height with the best fit to the molecular backscatter per channel is chosen
- Currently only done for FR-channels. NR reference heights are read from the config variables `refH_NR_{wavelength}`

In [ ]:
## Rayleigh-fit procedure --> produces the reference heights
data_cube.rayleighFit()

In [ ]:
## Display reference heights for a given cloud free group
grpIdx = 0
print(f"""Reference heights in meters for cloud free period {grpIdx} {data_cube.retrievals_highres['time64'][data_cube.clFreeGrps[grpIdx]]}:
355 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_FR"]['refHeight'], 0)}
532 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_FR"]['refHeight'], 0)}
1064 total FR: {np.round(data_cube.retrievals_profile['refH'][grpIdx]["1064_total_FR"]['refHeight'], 0)}
355 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_NR"]['refHeight'], 0)}
532 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_NR"]['refHeight'], 0)}""")

## GHK-Transmission Correction

In [ ]:
## Molecular polarization calibration 
data_cube.polarizationCaliMol()

In [ ]:
## Apply GHK-transmission correction
data_cube.transCor()

In [ ]:
## Aggregate GHK-transmission corrected profiles
data_cube.aggregate_profiles(var='sigTCor')
data_cube.aggregate_profiles(var='BGTCor')

## Klett and Raman retrieval

Produces the following profiles per channel:
- Klett: Aerosol Backscatter and Extinction
- Raman: Aerosol Backscatter, Aerosol Extinction, and Lidar Ratio

If `nr=True`, perform the retrievals for FR and NR channels. Otherwise only FR.

In [ ]:
## Klett retrieval for GHK-transmisson corrected profiles
data_cube.retrievalKlett(nr=True)

In [ ]:
## Raman retrieval for GHK-transmisson corrected profiles
data_cube.retrievalRaman(nr=True)

## Overlap Correction

Two methods are available for calculating the Overlap Function:

1. FRNR method
2. Raman method

And four methods (currently only 3 implemented) are available for applying the Overlap Correction:

0. no overlap correction
1. overlap correction with using the default overlap function (read function from file)
2. overlap correction with using the calculated overlap function
3. overlap correction with gluing near-range and far-range signal -> Not implemented yet!


In [ ]:
## Calculate overlap function
data_cube.overlapCalc()

## Fix spike in lower bins
data_cube.overlapFixLowestBins()

## Apply overlap correction
data_cube.overlapCor()

In [ ]:
## Aggregate overlap corrected profiles
data_cube.aggregate_profiles('sigOLCor')
data_cube.aggregate_profiles('BGOLCor')

In [ ]:
## Klett retrieval for overlap corrected profiles
data_cube.retrievalKlett(oc=True)

## Raman retrieval for overlap corrected profiles
data_cube.retrievalRaman(oc=True)

## Depol and Ångström Profiles

Retrieval of Volume and Particle depolarization as well as Ångström 355/532 backscatter 532/1064 backscatter, and 355/532 Extinction

In [ ]:
## Volume and particle depolarization
data_cube.calcDepol()

In [ ]:
## Ångström ratios
data_cube.Angstroem()

## Lidar Calibration

- Lidar calibration constants (LC) are retieved for each channel at each cloud free period for both Klett and Raman retieved profiles
- All retrieved LCs are stored in the database
- If no LCs can be retrieved for a given channel, the LCs in the time range [24h before the measurment, 24h after the measurement] for the given channel included in the database will be used
- The optimal LC, ie. the one with the lowest standard deviation per channel is used for the processing.
- The following priority order is used when choosing the optimal LC
    1. Raman retrieved LC from data
    2. Klett retrieved LC from data
    3. Raman retrieved LC from database
    4. Klett retrieved LC from database

In [ ]:
## Lidar calcibration for both Klett and Raman retrival
data_cube.LidarCalibration(db_path=DATABASE_PATH)

In [ ]:
## Display Lidar calibration constants per channel
data_cube.LCused

In [ ]:
## Store calibration constants in database
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='LC', method='raman')
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='LC', method='klett')
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='DC')

In [ ]:
## Save retrived optical profiles
write_profile2nc_file(data_cube=data_cube, prod_ls=["profiles", "NR_profiles", "OC_profiles"], collect_debug=True)

## High Resoulution Retrievals

The following high resolution (30s) time-height data are retrieved: 

- Attenuated backscatter
- Volume depolarization
- Molecular backscatter and extinction
- Quality mask
- QuasiV1 and QuasiV2 retrievals
- Target categorization V1 and V2

In [ ]:
## Highres attenuated backscatter and volume depolarization
data_cube.attBsc_volDepol()

## Highres molecular signal
data_cube.molecularHighres()

In [ ]:
## Quality mask of signal
data_cube.estQualityMask()

In [ ]:
## QuasiV1 retrievals and Target categorization
data_cube.quasiV1()

In [ ]:
## QuasiV2 retrievals and Target categorization
data_cube.quasiV2()

In [ ]:
## Save highres retrivals
write2nc_file(data_cube=data_cube, prod_ls=["att_bsc", "NR_att_bsc", "OC_att_bsc", "vol_depol", "quasi_results", "quasi_results_V2", "target_classification", "target_classification_V2"])